# Name Hierarchy Levels — iGEM Teams (Mid & High)

Assigns globally unique, publication-ready names to the mid-level and high-level
topic groups for **iGEM Teams**. For each group the LLM sees the low-level
sub-topics (name + description) it contains, then returns one name per group via
OpenAI function calling. Prompts come from `prompts_hierarchy.yaml`.

> Run `get_topic_hierarchy.ipynb` (this folder) **first**.

**Updates** `teams_topic_name_hierarchy.tsv` in today's run folder
(`assets/<date>/04/`) with `mid_name`
and `high_name` columns.

In [1]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live one level below 04-topic_hierarchy/, where the aux/
# package and setup_run.py reside; add that folder to the import path.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
from aux.paths import OPENAI_MODEL, set_seed
from setup_run import setup
from aux.naming import (
    load_prompts, make_client, build_system_prompt,
    name_hierarchy_level, load_naming_inputs, save_named_hierarchy,
)

set_seed()

# ── CONFIG: iGEM Teams ──────────────────────────────────────────────────
PREFIX = "teams"

prompts = load_prompts()
client = make_client()
system_prompt = build_system_prompt(prompts)

RUN = setup(corpus=PREFIX)  # today's run folder: assets/<date>/04/
topic_names, hierarchy = load_naming_inputs(RUN, PREFIX)
print(f"{PREFIX}: {len(topic_names)} low-level topics | "
      f"mid groups {hierarchy[hierarchy['mid'] >= 0]['mid'].nunique()} | "
      f"high groups {hierarchy[hierarchy['high'] >= 0]['high'].nunique()}")

teams: 154 low-level topics | mid groups 31 | high groups 10


## 1. Name the mid- and high-level groups

In [3]:
mid_names = name_hierarchy_level(
    hierarchy, topic_names, level_col="mid", label="mid",
    client=client, system_prompt=system_prompt, model=OPENAI_MODEL,
)
high_names = name_hierarchy_level(
    hierarchy, topic_names, level_col="high", label="high",
    client=client, system_prompt=system_prompt, model=OPENAI_MODEL,
)

  Naming 31 mid groups via gpt-4.1-nano …
  ✓ 31 mid names assigned
  Naming 10 high groups via gpt-4.1-nano …
  ✓ 10 high names assigned


## 2. Add the name columns and save

In [4]:
hierarchy["mid_name"] = hierarchy["mid"].map(mid_names)
hierarchy["high_name"] = hierarchy["high"].map(high_names)
save_named_hierarchy(RUN, hierarchy, PREFIX)

print(f"Saved → {RUN.dir / f'{PREFIX}_topic_name_hierarchy.tsv'}")
hierarchy.head(10)

Saved → /Users/cristian/Desktop/GitHub/igem-synbio/assets/reports/teams_topic_name_hierarchy.tsv


,global_name,low,mid,high,mid_name,high_name
0,Gene Expression Patterning,0,0,0,Gene Regulation and Cellular Circuitry Enginee...,Gene Regulation and Cellular Patterning
1,Pest and Vector Genetic Control,1,1,1,Synthetic Biology for Environmental and Agricu...,Environmental and Agricultural Biotechnology
2,Plant Disease Synthetic Diagnostics,2,1,1,Synthetic Biology for Environmental and Agricu...,Environmental and Agricultural Biotechnology
3,Biofilm and Quorum Disruption,3,2,0,Microbial Communication and Computational Systems,Gene Regulation and Cellular Patterning
4,Synthetic Biology Design Automation,4,3,0,"Synthetic Biology Tools, Standardization, and ...",Gene Regulation and Cellular Patterning
5,Genetic Device Stability and Evolution,5,3,0,"Synthetic Biology Tools, Standardization, and ...",Gene Regulation and Cellular Patterning
6,Environmental Pollution Bioremediation,6,4,2,Environmental Bioremediation and Pollution Con...,Environmental Bioremediation and Resource Reco...
7,MicroRNA Detection Platforms,7,5,3,Biosensors and Diagnostic Technologies,Synthetic Diagnostics and Biosensing Technologies
8,Eukaryotic and Microbial Circuit Engineering,8,0,0,Gene Regulation and Cellular Circuitry Enginee...,Gene Regulation and Cellular Patterning
9,Genetic Control Tools and Systems,9,0,0,Gene Regulation and Cellular Circuitry Enginee...,Gene Regulation and Cellular Patterning


## 3. Summary

In [5]:
n_mid = hierarchy["mid_name"].notna().sum()
n_high = hierarchy["high_name"].notna().sum()
print(f"{PREFIX}: {n_mid} topics with mid_name ({hierarchy['mid_name'].dropna().nunique()} unique), "
      f"{n_high} with high_name ({hierarchy['high_name'].dropna().nunique()} unique)")

teams: 154 topics with mid_name (31 unique), 154 with high_name (10 unique)
